# Natural Resource Processing, Sample Selection, and Imputation

**Capstone Project — Moody's Ratings**  
*Pipeline step 3 of 4: From raw data to analysis-ready panel*

This notebook performs four tasks:

**Part A — Natural Resource Data Processing (Sections 1-7)**  
Constructs country-year-level natural resource variables from raw production, reserves, and price data. Includes unit standardisation, price merging, backfilling of production gaps using constant-share assumptions, classification into resource categories (hydrocarbons, subsoil metals, precious metals), and creation of dominance dummies.

**Part B — Sample Selection and Filtering (Section 8)**  
Merges the NR variables with the master panel, selects the analysis variables, applies the country exclusion list (based on missingness diagnostics from Step 2), and produces the filtered sample.

**Part C — Imputation (Sections 9-11)**  
Fills remaining gaps using two methods applied sequentially:
1. **Linear interpolation** within each country's time series (for gaps between observed values)
2. **KNN imputation** for residual missing values (cross-country, distance-weighted)

**Part D — Population Merge (Section 12)**  
Merges World Bank population data into the final master panel and NR dataset, producing the definitive output files.

**Inputs:**  
- `../rawdata/production_values_w_prices-EM.csv` (Emilio's production-price data)
- `../rawdata/NR_Production-Price-Reserves-LEO.csv` (Leonardo's NR compilation)
- `../rawdata/NR_final_LEO.csv` (Leonardo's cleaned NR data)
- `../rawdata/PopulationWDI.csv` (World Bank population, wide format)
- `intermediary/master_data_wide.csv` (from Step 1)

**Outputs:**  
- `intermediary/Master.csv` (final panel with population)
- `intermediary/NaturalResource.csv` (NR data with population)
- `intermediary/master_data_imputed.csv` (intermediate, pre-population)
- `intermediary/NRCleanData.csv` (intermediate, pre-population)

---

## 0. Setup

In [ ]:
import os
os.makedirs("intermediary", exist_ok=True)
os.makedirs("Graphics/NB3", exist_ok=True)

import pandas as pd
import numpy as np
from itertools import product
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

---

# Part A: Natural Resource Data Processing

## 1. Load Raw Data

Three source files contain production volumes, reserves, and prices for hydrocarbons and minerals. The data comes from multiple compilers (USGS, BP Statistical Review, national agencies) and requires harmonisation of resource names, country names, and units before merging.

In [ ]:
# Raw data files (relative paths from code/ directory)
nrp_em = pd.read_csv("../rawdata/production_values_w_prices-EM.csv", index_col=0)
nrp_l1 = pd.read_csv("../rawdata/NR_Production-Price-Reserves-LEO.csv")
nrp_l2 = pd.read_csv("../rawdata/NR_final_LEO.csv")

print(f"Emilio data: {nrp_em.shape}")
print(f"Leo data (with world): {nrp_l1.shape}")
print(f"Leo data (final): {nrp_l2.shape}")

## 2. Prepare Production, Reserves, and Price Data

We filter to 1995 onwards and to Production/Reserves metrics only (excluding Consumption). World/Global production totals are extracted separately as reference benchmarks for the backfilling procedure in Section 4. Two price series are merged, with Leo's prices as primary and Emilio's as fallback.

In [ ]:
# ── Country-level production and reserves ──
nrp_l2 = nrp_l2[
    (nrp_l2["Year"] >= 1995) & (nrp_l2["Metric"].isin(["Production", "Reserves"]))
][["Country", "Metric", "Year", "Resource", "Value"]]

# ── World/Global production totals (used as reference for backfilling) ──
nrp_world = nrp_l1[
    (nrp_l1["Country Code"] == "OWID_WRL") & nrp_l1["Metric"].isin(["Production", "Reserves"])
][["Country Name", "Metric", "Year", "Resource", "Value"]].copy()
nrp_world.rename(columns={"Country Name": "Country"}, inplace=True)

# Harmonise resource names across datasets
resource_name_map = {
    'Aluminum': 'Aluminium',
    'Platinum group metals': 'Platinum Group',
    'Rare earths': 'Rare Earth',
}
nrp_world['Resource'] = nrp_world['Resource'].replace(resource_name_map)

# ── Price series 1 (from Leo's data: unit values) ──
price_l = nrp_l1[
    (nrp_l1["Year"] >= 1995) & (nrp_l1["Metric"] == "Unit Value")
][["Year", "Resource", "Value"]].copy()
price_l.rename(columns={"Value": "Price"}, inplace=True)
price_l['Resource'] = price_l['Resource'].replace(resource_name_map)

# ── Price series 2 (from Emilio's data) ──
nrp_em = nrp_em[nrp_em["Year"] >= 1995][["Country", "Year", "Resource", "Price"]].copy()
nrp_em.rename(columns={"Price": "Price2"}, inplace=True)
nrp_em_prices = nrp_em[["Year", "Resource", "Price2"]].drop_duplicates()

# ── Merge price series (Price1 primary, Price2 as fallback) ──
prices = pd.merge(price_l, nrp_em_prices, on=["Year", "Resource"], how="outer")
prices["Price"] = prices["Price"].fillna(prices["Price2"])
prices['Resource'] = prices['Resource'].replace(resource_name_map)
prices.drop_duplicates(inplace=True)

print(f"Country-level NR rows: {len(nrp_l2):,}")
print(f"World reference rows: {len(nrp_world):,}")
print(f"Price observations: {len(prices):,}")
print(f"Resources with prices: {prices['Resource'].nunique()}")

## 3. Merge Production with Prices and Clean

Production volumes are joined with prices to compute total production values. We also standardise country names to World Bank conventions, remove non-country entities (OPEC, regional aggregates), and drop resources that are duplicated or have insufficient coverage (Petroleum, Platinum Group).

In [ ]:
# Combine country and world production
nrp = pd.concat([nrp_l2, nrp_world], ignore_index=True)

# Merge with prices
nrpv = pd.merge(nrp, prices[["Year", "Resource", "Price"]], on=["Year", "Resource"], how="left")
nrpv = nrpv[nrpv["Year"] >= 1995]

# Remove duplicate resources (Petroleum overlaps with Oil; Platinum Group has poor coverage)
nrpv = nrpv[~nrpv["Resource"].isin(["Petroleum", "Platinum Group"])]

# Standardise country names to World Bank conventions
country_name_map = {
    'Venezuela: Orinoco Belt': 'Venezuela, RB',
    "US": "United States",
    "Congo": "Congo, Rep.",
    "Democratic Republic of Congo": "Congo, Dem. Rep.",
    "DR Congo": "Congo, Dem. Rep.",
    "Republic of Congo ": "Congo, Rep.",
    "Republic of Congo": "Congo, Rep.",
    "Brunei": "Brunei Darussalam",
    "Burma": "Myanmar",
    "Czech Republic": "Czechia",
    "Egypt": "Egypt, Arab Rep.",
    "Iran": "Iran, Islamic Rep.",
    "Kyrgyzstan": "Kyrgyz Republic",
    "Laos": "Lao PDR",
    "South Korea": "Korea, Rep.",
    "Slovakia": "Slovak Republic",
    "North Korea": "Korea, Dem. People's Rep.",
    "Syria": "Syrian Arab Republic",
    "Trinidad & Tobago": "Trinidad and Tobago",
    "Turkey": "Turkiye",
    "Venezuela": "Venezuela, RB",
    "Yemen": "Yemen, Rep.",
    "Vietnam": "Viet Nam",
}
nrpv["Country"] = nrpv["Country"].replace(country_name_map)

# Remove non-country entities
non_country_entities = [
    '                 OPEC',
    '                 Non-OPEC ',
    'Central America',
    'Eastern Africa',
    'Middle Africa',
    'Western Africa',
]
nrpv = nrpv[~nrpv["Country"].isin(non_country_entities)]
nrpv = nrpv.drop_duplicates()

print(f"Merged NR data: {nrpv.shape[0]:,} rows")
print(f"Resources: {sorted(nrpv['Resource'].unique())}")
print(f"Countries: {nrpv['Country'].nunique()}")

## 4. Backfill Production Gaps for Selected Minerals

Some minerals (Tin, Nickel, Vanadium, Manganese, Cadmium, Magnesium compounds, Iron ore) have incomplete country-level coverage in early years. For each mineral, we identify the earliest year where country-level data accounts for over 70% of global/world production. We then backfill earlier years by applying each country's production share at that threshold year to the known global totals.

This constant-share assumption is a simplification, but it preserves cross-country relativities and avoids introducing zeros for countries that were producing but not reporting.

In [ ]:
target_minerals = ["Tin", "Nickel", "Vanadium", "Manganese", "Cadmium",
                   "Magnesium compounds", "Iron ore"]

prod = nrpv[nrpv["Metric"] == "Production"].copy()

# ── Reference production (Global preferred, World as fallback) ──
global_prod = prod[prod["Country"] == "Global"][["Year", "Resource", "Value"]].copy()
global_prod.rename(columns={"Value": "Global_Production"}, inplace=True)

world_prod = prod[prod["Country"] == "World"][["Year", "Resource", "Value"]].copy()
world_prod.rename(columns={"Value": "World_Production"}, inplace=True)

reference_prod = pd.merge(global_prod, world_prod, on=["Year", "Resource"], how="outer")
reference_prod["Reference_Production"] = (
    reference_prod["Global_Production"].fillna(reference_prod["World_Production"])
)

# ── Country shares and coverage ──
country_prod = prod[~prod["Country"].isin(["World", "Global"])].copy()
country_with_ref = pd.merge(
    country_prod[["Country", "Year", "Resource", "Value"]],
    reference_prod[["Year", "Resource", "Reference_Production"]],
    on=["Year", "Resource"], how="left",
)
country_with_ref["Country_Share"] = (
    country_with_ref["Value"] / country_with_ref["Reference_Production"]
)

# Total country coverage per resource-year
country_total = country_prod.groupby(["Year", "Resource"])["Value"].sum().reset_index()
country_total.rename(columns={"Value": "Country_Total"}, inplace=True)
country_total = pd.merge(
    country_total, reference_prod[["Year", "Resource", "Reference_Production"]],
    on=["Year", "Resource"], how="left",
)
country_total["Coverage"] = (
    country_total["Country_Total"] / country_total["Reference_Production"]
) * 100

# ── Find threshold year (earliest year with >70% coverage) per mineral ──
threshold_years = {}
constant_shares = {}

for mineral in target_minerals:
    mineral_coverage = country_total[
        (country_total["Resource"] == mineral) & (country_total["Coverage"] > 70)
    ].sort_values("Year")

    if len(mineral_coverage) > 0:
        threshold_year = mineral_coverage["Year"].min()
        threshold_years[mineral] = threshold_year

        # Country shares at threshold year
        shares_at_threshold = country_with_ref[
            (country_with_ref["Resource"] == mineral)
            & (country_with_ref["Year"] == threshold_year)
            & (country_with_ref["Value"] > 0)
        ][["Country", "Country_Share"]].copy()
        constant_shares[mineral] = shares_at_threshold
    else:
        threshold_years[mineral] = None
        print(f"WARNING: {mineral} never reaches 70% coverage")

print("Threshold years (earliest year with >70% country coverage):")
for mineral, year in threshold_years.items():
    n_countries = len(constant_shares.get(mineral, []))
    print(f"  {mineral}: {year} ({n_countries} producing countries)")

In [ ]:
# ── Generate backfilled rows ──
backfilled_rows = []

for mineral, threshold_year in threshold_years.items():
    if threshold_year is None:
        continue

    years_to_backfill = sorted([y for y in prod["Year"].unique() if y < threshold_year])
    ref_for_backfill = reference_prod[
        (reference_prod["Resource"] == mineral)
        & (reference_prod["Year"].isin(years_to_backfill))
    ][["Year", "Reference_Production"]].copy()

    shares = constant_shares[mineral]

    for _, ref_row in ref_for_backfill.iterrows():
        year = ref_row["Year"]
        ref_prod_value = ref_row["Reference_Production"]
        if pd.isna(ref_prod_value):
            continue

        for _, share_row in shares.iterrows():
            backfilled_rows.append({
                "Country": share_row["Country"],
                "Year": year,
                "Resource": mineral,
                "Value": ref_prod_value * share_row["Country_Share"],
                "Metric": "Production",
                "Source": "Backfilled",
            })

backfilled_df = pd.DataFrame(backfilled_rows)
print(f"Backfilled rows created: {len(backfilled_df):,}")
print(f"\nBy mineral:")
print(backfilled_df.groupby("Resource").agg(
    Years=("Year", lambda x: f"{x.min()}-{x.max()}"),
    Countries=("Country", "nunique"),
    Rows=("Year", "count"),
))

# ── Integrate into main dataset ──
nrpv_clean = nrpv[~nrpv["Country"].isin(["World", "Global"])].copy()
nrpv_clean["Source"] = "Original"

backfilled_for_merge = backfilled_df.copy()
backfilled_for_merge = pd.merge(
    backfilled_for_merge, prices[["Year", "Resource", "Price"]],
    on=["Year", "Resource"], how="left",
)
backfilled_for_merge = backfilled_for_merge[list(nrpv_clean.columns)]

# Remove original partial data for target minerals before threshold year
for mineral, threshold_year in threshold_years.items():
    if threshold_year is None:
        continue
    mask = ((nrpv_clean["Resource"] == mineral)
            & (nrpv_clean["Year"] < threshold_year)
            & (nrpv_clean["Metric"] == "Production"))
    nrpv_clean = nrpv_clean[~mask]

nrpv = pd.concat([nrpv_clean, backfilled_for_merge], ignore_index=True)
nrpv = nrpv.sort_values(["Resource", "Country", "Year", "Metric"]).reset_index(drop=True)

print(f"\nFinal NR dataset: {nrpv.shape[0]:,} rows")

## 5. IEA Price Adjustments (2021)

For Copper, Nickel, and Rare Earth, IEA-sourced year-over-year price changes are used to estimate 2021 prices from observed 2020 prices, since 2021 commodity prices were not yet available in the original dataset.

In [ ]:
price_changes_2021 = {
    "Copper": 44.82477588,       # Base Metals (IEA)
    "Nickel": 50.13077594,       # Battery Metals (IEA)
    "Rare Earth": 90.52167524,   # Rare Earth (IEA)
}

for resource, pct_change in price_changes_2021.items():
    mask_2021 = (nrpv["Resource"] == resource) & (nrpv["Year"] == 2021)
    mask_2020 = (nrpv["Resource"] == resource) & (nrpv["Year"] == 2020)

    price_2020 = nrpv.loc[mask_2020, "Price"].values
    if len(price_2020) > 0:
        price_2021 = price_2020[0] * (1 + pct_change / 100)
        nrpv.loc[mask_2021, "Price"] = price_2021
        print(f"{resource}: 2020 price = {price_2020[0]:,.2f}, "
              f"YoY = +{pct_change:.1f}%, 2021 price = {price_2021:,.2f}")

## 6. Compute Production and Reserves Values

With prices attached, we compute total production value (Volume x Price) and total reserves value for each country-resource-year observation. Resources are classified into three categories, and dominance dummies indicate whether a country derives more than 50% of its total production value from a single category.

In [ ]:
# Filter to pre-2022 and remove remaining problematic resources
nrpv = nrpv[
    (nrpv["Year"] < 2022)
    & (~nrpv["Resource"].isin(["Petroleum", "Platinum group metals", "Iron ore"]))
]

# Pivot to have Production and Reserves as columns
nrpv = nrpv.pivot_table(
    index=["Country", "Year", "Resource", "Source", "Price"],
    columns="Metric", values="Value",
).reset_index()

nrpv["Production_TotalValue"] = nrpv["Production"] * nrpv["Price"]
nrpv["Reserves_TotalValue"] = nrpv["Reserves"] * nrpv["Price"]

# ── Classify resources ──
hydrocarbons = ['Oil', 'Natural Gas', 'Coal']
precious_metals = ['Gold', 'Silver']
subsoil = [
    'Bauxite', 'Copper', 'Aluminium', 'Lead', 'Lithium',
    'Zinc', 'Cadmium', 'Cobalt', 'Iron ore', 'Magnesium compounds',
    'Manganese', 'Nickel', 'Rare earths', 'Tin', 'Vanadium',
    'Natural Graphite',
]

def classify_resource(x):
    if x in hydrocarbons:
        return 'Hydrocarbons'
    elif x in subsoil:
        return 'Subsoil Metals'
    elif x in precious_metals:
        return 'Precious Metals'
    else:
        return 'Others'

nrpv['Resource Category'] = nrpv['Resource'].apply(classify_resource)
print(f"NR with values: {nrpv.shape[0]:,} rows")
print(f"Resource categories: {nrpv['Resource Category'].value_counts().to_dict()}")

## 7. Aggregate to Country-Year Level

Production values are summed by resource category and country-year. Dominance dummies are created: a country is classified as hydrocarbon-dominant, subsoil-metal-dominant, or precious-metal-dominant if that category accounts for more than 50% of its total production value in a given year.

In [ ]:
# Total production value by country, year, resource category
category_totals = (
    nrpv.groupby(['Country', 'Year', 'Resource Category'])['Production_TotalValue']
    .sum().reset_index()
)

# Total across all categories
country_year_totals = (
    nrpv.groupby(['Country', 'Year'])['Production_TotalValue']
    .sum().reset_index()
    .rename(columns={'Production_TotalValue': 'Total_Production_Value'})
)

# Pivot categories to columns
category_pivot = category_totals.pivot_table(
    index=['Country', 'Year'], columns='Resource Category',
    values='Production_TotalValue', fill_value=0,
).reset_index()

merged = category_pivot.merge(country_year_totals, on=['Country', 'Year'])

# Dominance dummies (>50% share)
merged['Hydrocarbons_Dominant'] = (
    merged.get('Hydrocarbons', 0) / merged['Total_Production_Value'] > 0.5
).astype(int)
merged['Subsoil_Metals_Dominant'] = (
    merged.get('Subsoil Metals', 0) / merged['Total_Production_Value'] > 0.5
).astype(int)
merged['Precious_Metals_Dominant'] = (
    merged.get('Precious Metals', 0) / merged['Total_Production_Value'] > 0.5
).astype(int)

# Aggregate reserves and total production quantities
reserves_totals = nrpv.groupby(['Country', 'Year']).agg({
    'Reserves_TotalValue': 'sum',
    'Reserves': 'sum',
    'Production': 'sum',
}).reset_index().rename(columns={
    'Reserves_TotalValue': 'Total_Reserves_Value',
    'Reserves': 'Total_Reserves',
    'Production': 'Total_Production',
})

# Final NR aggregated dataset
nrpa = merged[['Country', 'Year', 'Total_Production_Value']].merge(
    reserves_totals, on=['Country', 'Year'], how='left',
)
nrpa['Hydrocarbons_Dominant'] = merged['Hydrocarbons_Dominant']
nrpa['Subsoil_Metals_Dominant'] = merged['Subsoil_Metals_Dominant']
nrpa['Precious_Metals_Dominant'] = merged['Precious_Metals_Dominant']

nrpa = nrpa[['Country', 'Year', 'Total_Production', 'Total_Reserves',
             'Total_Production_Value', 'Total_Reserves_Value',
             'Hydrocarbons_Dominant', 'Subsoil_Metals_Dominant',
             'Precious_Metals_Dominant']]

feasible_countries = nrpa["Country"].tolist()
print(f"NR aggregated data: {nrpa.shape[0]:,} rows")
print(f"Countries with NR data: {nrpa['Country'].nunique()}")

---

# Part B: Sample Selection and Variable Filtering

## 8. Merge with Master Data and Apply Filters

The NR variables are merged with the master panel from Step 1. We then:
1. Select the subset of analysis variables (informed by the missingness diagnostics in Step 2)
2. Restrict to countries present in the NR dataset (`feasible_countries`)
3. Exclude countries with structurally excessive missingness (identified iteratively through the diagnostics)
4. Fill structural zeros for IMF credit and death rates

In [ ]:
# Load master data
df = pd.read_csv("intermediary/master_data_wide.csv")

# Merge with NR aggregated variables
df = pd.merge(df, nrpa, left_on=["Country Name", "Year"], right_on=["Country", "Year"], how="left")

# ── Variable selection ──
keep_vars = [
    'Country Code', 'Country Name', 'Year',
    # Governance
    'Access to electricity (% of population)',
    'Adjusted savings: gross savings (% of GNI)',
    'Agriculture',
    'Capital depreciation rate',
    'Clientelism index',
    'Death rates, crude per 1000 people',
    'Domestic credit to private sector (% of GDP)',
    'Economic Complexity Index',
    'GDP per capita (constant prices, PPP)',
    'Government revenue',
    'Gross fixed capital formation, all, Constant prices, Percent of GDP',
    'Human capital index',
    'Industry',
    'Inflation, consumer prices (annual %)',
    'Landlocked',
    'Lending interest rate (%)',
    'Life expectancy at birth, total (years)',
    'Manufacturing',
    'Mineral rents (% of GDP)',
    'Mobile cellular subscriptions (per 100 people)',
    'Natural gas rents (% of GDP)',
    'Oil rents (% of GDP)',
    'Political corruption index',
    'Political stability — estimate',
    'Primary net lending, General government, Percent of GDP',
    'Property rights',
    'Real interest rate (%)',
    'Rule of law index',
    'Services',
    'Share of consumption in GDP',
    'Share of government spending in GDP',
    'Share of investment in GDP',
    'Total natural resources rents (% of GDP)',
    'Trade (% of GDP)',
    'Urban population (% of total population)',
    'Use of IMF credit (DOD, current US$)',
    # NR variables
    'Total_Production',
    'Total_Reserves',
    'Total_Production_Value',
    'Total_Reserves_Value',
    'Hydrocarbons_Dominant',
    'Subsoil_Metals_Dominant',
    'Precious_Metals_Dominant',
]

# ── Country exclusion list ──
# Based on iterative missingness analysis (see Step 2 diagnostics)
omit_countries = [
    'BDI', 'BTN', 'CAF', 'ERI', 'FJI', 'GUY', 'ISL', 'MNE', 'NCL',
    'PRK', 'SLB', 'SLE', 'SSD', 'SUR', 'SYR', 'GUF', 'TWN', 'CUB',
    'TLS', 'AFG', 'TKM', 'KHM', 'XKX', 'BEN',
]

# Apply filters
df = df[keep_vars]
df = df[df["Country Name"].isin(feasible_countries)]
df = df[~df["Country Code"].isin(omit_countries)]

# Fill structural zeros
df["Use of IMF credit (DOD, current US$)"] = df["Use of IMF credit (DOD, current US$)"].fillna(0)
df["Death rates, crude per 1000 people"] = df["Death rates, crude per 1000 people"].fillna(0)

cmaster = df.copy()

print(f"Filtered sample: {cmaster.shape[0]:,} rows")
print(f"Countries: {cmaster['Country Code'].nunique()}")
print(f"Variables: {cmaster.shape[1] - 3} (excl. identifiers)")
print(f"Remaining missing cells: {cmaster.iloc[:, 3:].isna().sum().sum():,}")

---

# Part C: Imputation

## 9. Linear Interpolation (Within-Country)

The first imputation pass uses linear interpolation within each country's time series. For each variable, if a country has at least one observed value, missing values between observed points are filled by assuming a constant rate of change. Interpolation is applied in both directions (`limit_direction='both'`), which also extrapolates slightly at the edges of a country's time series.

Variables excluded from imputation: `Landlocked` (time-invariant dummy), dominance dummies, and total reserves/production values (which should remain as-is or be treated separately).

In [ ]:
EXCLUDE_VARS = [
    'Landlocked',
    'Subsoil_Metals_Dominant',
    'Hydrocarbons_Dominant',
    'Precious_Metals_Dominant',
    'Total_Reserves_Value',
    'Total_Reserves',
]


def impute_linear_interpolation(df):
    """
    Impute missing values using linear interpolation within each country.
    Only fills gaps where the country has at least one observation for the variable.
    """
    id_cols = ['Country Code', 'Country Name', 'Year']
    data_cols = [c for c in df.columns if c not in id_cols and c not in EXCLUDE_VARS]

    pre_missing = df[data_cols].isna().sum().sum()
    print("=" * 70)
    print("IMPUTATION STEP 1: Linear interpolation by country")
    print("=" * 70)
    print(f"\nVariables excluded: {[c for c in EXCLUDE_VARS if c in df.columns]}")
    print(f"Variables to impute: {len(data_cols)}")
    print(f"Total missing cells before: {pre_missing:,}")

    df_sorted = df.sort_values(['Country Code', 'Year']).reset_index(drop=True)

    imputed_dfs = []
    for country in df_sorted['Country Code'].unique():
        country_data = df_sorted[df_sorted['Country Code'] == country].copy()
        for col in data_cols:
            if country_data[col].notna().any():
                country_data[col] = country_data[col].interpolate(
                    method='linear', limit_direction='both'
                )
        imputed_dfs.append(country_data)

    df_imputed = pd.concat(imputed_dfs, ignore_index=True)

    post_missing = df_imputed[data_cols].isna().sum().sum()
    print(f"Total missing cells after: {post_missing:,}")
    print(f"Cells filled: {pre_missing - post_missing:,} "
          f"({100 * (pre_missing - post_missing) / pre_missing:.1f}%)")

    remaining = df_imputed[data_cols].isna().sum()
    remaining = remaining[remaining > 0].sort_values(ascending=False)
    if len(remaining) > 0:
        print(f"\nVariables with remaining missing (country has no data at all):")
        for var, count in remaining.items():
            print(f"   {var}: {count} cells")
    else:
        print("\nAll missing values filled.")

    return df_imputed


df_imputed = impute_linear_interpolation(cmaster)

## 10. KNN Imputation (Cross-Country)

After linear interpolation, some cells may still be missing if a country has no data at all for a particular variable. These are filled using K-Nearest Neighbours imputation (k=5, distance-weighted), which finds the most similar countries in the feature space and uses their values.

Before applying KNN to the full dataset, we run a validation test: we randomly mask 5% of known values, impute them, and compare predictions to actuals. This provides an estimate of imputation quality.

In [ ]:
# Extended exclusion list for KNN (includes production totals)
EXCLUDE_VARS_KNN = [
    'Landlocked',
    'Subsoil_Metals_Dominant',
    'Hydrocarbons_Dominant',
    'Precious_Metals_Dominant',
    'Total_Production_Value',
    'Total_Reserves_Value',
    'Total_Production',
    'Total_Reserves',
]


def validate_knn_imputer(df, test_fraction=0.05, n_neighbors=5, random_state=42):
    """
    Validate KNN imputation by masking 5% of known values, imputing them,
    and comparing predictions to actuals.
    """
    id_cols = ['Country Code', 'Country Name', 'Year']
    numeric_cols = [c for c in df.columns
                    if c not in id_cols and c not in EXCLUDE_VARS_KNN
                    and df[c].dtype in ['float64', 'int64']]

    print("=" * 70)
    print("KNN IMPUTER VALIDATION TEST")
    print("=" * 70)
    print(f"\nVariables to impute: {len(numeric_cols)}")
    print(f"n_neighbors: {n_neighbors}")

    df_numeric = df[numeric_cols].copy()

    # Randomly mask known values
    np.random.seed(random_state)
    non_missing_mask = ~df_numeric.isna()
    non_missing_indices = list(zip(*np.where(non_missing_mask)))
    n_to_mask = int(len(non_missing_indices) * test_fraction)
    test_indices = np.random.choice(len(non_missing_indices), size=n_to_mask, replace=False)
    test_positions = [non_missing_indices[i] for i in test_indices]

    print(f"Total non-missing cells: {len(non_missing_indices):,}")
    print(f"Cells masked for testing: {n_to_mask:,} ({test_fraction * 100:.0f}%)")

    # Store true values
    true_values = []
    for row, col in test_positions:
        true_values.append({
            'row': row, 'col': col,
            'variable': numeric_cols[col],
            'true_value': df_numeric.iloc[row, col],
        })

    # Create masked dataset
    df_masked = df_numeric.copy()
    for row, col in test_positions:
        df_masked.iloc[row, col] = np.nan

    # Scale, impute, inverse-transform
    scaler = StandardScaler()
    df_scaled = pd.DataFrame(
        scaler.fit_transform(df_masked.fillna(df_masked.mean())),
        columns=numeric_cols,
    )
    df_scaled[df_masked.isna()] = np.nan

    imputer = KNNImputer(n_neighbors=n_neighbors, weights='distance')
    df_imputed_scaled = imputer.fit_transform(df_scaled)
    df_result = pd.DataFrame(
        scaler.inverse_transform(df_imputed_scaled), columns=numeric_cols,
    )

    # Evaluate
    for item in true_values:
        item['predicted_value'] = df_result.iloc[item['row'], item['col']]
        item['error'] = item['predicted_value'] - item['true_value']
        item['abs_error'] = abs(item['error'])
        item['pct_error'] = (
            100 * item['abs_error'] / abs(item['true_value'])
            if item['true_value'] != 0 else np.nan
        )

    results_df = pd.DataFrame(true_values)
    y_true = results_df['true_value'].values
    y_pred = results_df['predicted_value'].values

    print(f"\nR2 Score: {r2_score(y_true, y_pred):.4f}")
    print(f"MAE: {mean_absolute_error(y_true, y_pred):.4f}")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_true, y_pred)):.4f}")
    print(f"Median % Error: {results_df['pct_error'].median():.2f}%")

    var_results = results_df.groupby('variable').agg({
        'true_value': 'count',
        'abs_error': 'mean',
        'pct_error': 'median',
    }).rename(columns={
        'true_value': 'n_tested',
        'abs_error': 'mean_abs_error',
        'pct_error': 'median_pct_error',
    }).sort_values('median_pct_error', ascending=False)

    print("\nVariables with HIGHEST error:")
    print(var_results.head(10).to_string())
    print("\nVariables with LOWEST error:")
    print(var_results.tail(10).to_string())

    return results_df, var_results


results_df, var_results = validate_knn_imputer(df_imputed, test_fraction=0.05, n_neighbors=5)

In [ ]:
def apply_knn_imputer(df, n_neighbors=5):
    """Apply KNN imputation to fill all remaining missing values."""
    id_cols = ['Country Code', 'Country Name', 'Year']
    numeric_cols = [c for c in df.columns
                    if c not in id_cols and c not in EXCLUDE_VARS_KNN
                    and df[c].dtype in ['float64', 'int64']]

    print("=" * 70)
    print("IMPUTATION STEP 2: KNN imputation (cross-country)")
    print("=" * 70)

    pre_missing = df[numeric_cols].isna().sum().sum()
    print(f"\nMissing before: {pre_missing:,}")

    # Scale
    scaler = StandardScaler()
    df_scaled = pd.DataFrame(
        scaler.fit_transform(df[numeric_cols].fillna(df[numeric_cols].mean())),
        columns=numeric_cols,
    )
    df_scaled[df[numeric_cols].isna()] = np.nan

    # Impute
    imputer = KNNImputer(n_neighbors=n_neighbors, weights='distance')
    imputed_array = imputer.fit_transform(df_scaled)

    # Inverse transform
    imputed_values = scaler.inverse_transform(imputed_array)

    df_final = df.copy()
    df_final[numeric_cols] = imputed_values

    post_missing = df_final[numeric_cols].isna().sum().sum()
    print(f"Missing after: {post_missing:,}")
    print(f"Filled: {pre_missing - post_missing:,} cells")

    excluded_in_data = [c for c in EXCLUDE_VARS_KNN if c in df.columns]
    if excluded_in_data:
        excluded_missing = df_final[excluded_in_data].isna().sum().sum()
        print(f"\nExcluded variables (not imputed): {excluded_in_data}")
        print(f"Missing in excluded vars: {excluded_missing}")

    return df_final


cmaster_imp = apply_knn_imputer(df_imputed, n_neighbors=5)

## 11. Save Intermediate Outputs

In [ ]:
cmaster_imp.to_csv("intermediary/master_data_imputed.csv")
nrpv.to_csv("intermediary/NRCleanData.csv")

print("Intermediate outputs saved:")
print("  intermediary/master_data_imputed.csv")
print("  intermediary/NRCleanData.csv")

## 12. Population Data and Final Master Files

The last step merges World Bank population data into both the master panel and the NR dataset. Population figures are needed to compute per-capita resource production and reserves values used in the descriptive statistics and regressions.

The population data comes from the World Development Indicators (WDI) in wide format, with years encoded as `"1995 [YR1995]"`. We reshape it to long format and merge on country-year.

In [ ]:
# ── Load and reshape WDI population data ──
df_pop = pd.read_csv("../rawdata/PopulationWDI.csv")

# Remove metadata rows (NaN country codes)
df_pop = df_pop[df_pop["Country Code"].notna()].copy()

id_cols = ["Country Name", "Country Code"]
year_cols = [col for col in df_pop.columns if "[YR" in col]

# Wide to long
df_long = pd.melt(
    df_pop,
    id_vars=id_cols,
    value_vars=year_cols,
    var_name="Year_raw",
    value_name="Population",
)

# Extract year from "1995 [YR1995]" format
df_long["Year"] = df_long["Year_raw"].str.extract(r"(\d{4})").astype(int)
df_long = df_long.drop(columns=["Year_raw"])

# Clean population values (handles ".." entries)
df_long["Population"] = pd.to_numeric(df_long["Population"], errors="coerce")
df_long = df_long[["Country Name", "Country Code", "Year", "Population"]]
df_long = df_long.sort_values(["Country Name", "Year"]).reset_index(drop=True)

print(f"Population data: {df_long.shape[0]:,} rows")
print(f"Year range: {df_long['Year'].min()}-{df_long['Year'].max()}")
print(f"Countries: {df_long['Country Code'].nunique()}")
print(f"Missing values: {df_long['Population'].isna().sum()}")

In [ ]:
# ── Merge population into master panel ──
master = pd.read_csv("intermediary/master_data_imputed.csv", index_col=0)
master = master.merge(
    df_long[["Country Code", "Year", "Population"]],
    on=["Country Code", "Year"],
    how="left",
)

# ── Merge population into NR dataset ──
nr = nrpv.merge(
    master[["Country Name", "Country Code"]],
    left_on="Country", right_on="Country Name",
    how="left",
)
nr = nr.merge(
    df_long[["Country Code", "Year", "Population"]],
    on=["Country Code", "Year"],
    how="left",
)

# ── Save final outputs ──
master.to_csv("intermediary/Master.csv")
nr.to_csv("intermediary/NaturalResource.csv")

print("Final outputs saved:")
print(f"  intermediary/Master.csv ({master.shape})")
print(f"  intermediary/NaturalResource.csv ({nr.shape})")
print(f"\nPopulation coverage in master: "
      f"{master['Population'].notna().sum()}/{len(master)} rows")

---

## Summary

| Stage | Description |
|-------|-------------|
| **NR Processing** | Merged 3 raw data sources, standardised names, backfilled 7 minerals using constant-share method, applied 2021 IEA price adjustments, classified into 3 resource categories |
| **Sample Selection** | 43 variables retained, countries filtered to those with NR data, 24 countries excluded for data quality |
| **Imputation Stage 1** | Linear interpolation within each country's time series (fills interior gaps) |
| **Imputation Stage 2** | KNN imputation (k=5, distance-weighted) for remaining gaps, validated with 5% holdout test |
| **Population Merge** | WDI population data merged into master panel and NR dataset |
| **Outputs** | `Master.csv` (final panel with population), `NaturalResource.csv` (NR data with population), plus intermediate `master_data_imputed.csv` and `NRCleanData.csv` |

**Pipeline position:** Step 3 of 4:  
`1_cleaning_master_data` -> `2_MissingCheck` -> **`3_Imputing`** -> `4_Clustering`

In [ ]:
# ── NB3 SUMMARY ──
print("=" * 70)
print("NB3: IMPUTATION SUMMARY")
print("=" * 70)

print(f"\n--- Natural resource data ---")
print(f"Resources in final NR data: {nrpv['Resource'].nunique()}")
print(f"Countries in NR aggregates: {nrpa['Country'].nunique()}")
print(f"NR data rows: {nrpv.shape[0]:,}")

print(f"\n--- Sample selection ---")
print(f"Countries in filtered sample: {cmaster['Country Code'].nunique()}")
print(f"Variables (excl. identifiers): {cmaster.shape[1] - 3}")
print(f"Rows: {len(cmaster):,}")

print(f"\n--- Imputation results ---")
id_cols = ['Country Code', 'Country Name', 'Year']
data_cols = [c for c in cmaster_imp.columns if c not in id_cols]
remaining = cmaster_imp[data_cols].isna().sum()
remaining = remaining[remaining > 0]
if len(remaining) > 0:
    print(f"Variables still missing after imputation: {len(remaining)}")
    for v, n in remaining.items():
        print(f"  {v}: {n}")
else:
    print("All cells filled. Zero missing values.")

print(f"\n--- Final outputs ---")
print(f"Master.csv: {master.shape} (with Population)")
print(f"  Population coverage: {master['Population'].notna().sum()}/{len(master)}")
print(f"NaturalResource.csv: {nr.shape}")
print(f"\nFiles saved to intermediary/:")
print(f"  master_data_imputed.csv, NRCleanData.csv, Master.csv, NaturalResource.csv")

In [ ]:
# ── NB3 SUMMARY ──
print("=" * 70)
print("NB3: IMPUTATION PIPELINE SUMMARY")
print("=" * 70)

print(f"\n--- Natural resource data ---")
print(f"Resources: {nrpv['Resource'].nunique()}")
print(f"Countries in NR data: {nrpv['Country'].nunique()}")
print(f"NR rows: {nrpv.shape[0]:,}")

print(f"\n--- Sample selection ---")
print(f"Final sample: {cmaster_imp['Country Code'].nunique()} countries x {cmaster_imp['Year'].nunique()} years")
print(f"Variables: {cmaster_imp.shape[1] - 3} (excl. identifiers)")
print(f"Omitted countries: {len(omit_countries)}")

print(f"\n--- Imputation ---")
remaining = cmaster_imp.select_dtypes(include='number').isna().sum().sum()
print(f"Missing cells after full imputation: {remaining}")

print(f"\n--- Population merge ---")
print(f"Master.csv: {master.shape}")
print(f"Population coverage: {master['Population'].notna().sum()}/{len(master)} rows")

print(f"\nSaved:")
print(f"  intermediary/master_data_imputed.csv")
print(f"  intermediary/NRCleanData.csv")
print(f"  intermediary/Master.csv")
print(f"  intermediary/NaturalResource.csv")